# Oturum 6 — Bentopy: Kalabalık Hücresel Sistemler

**Biyofizik 2026 Kursu · Dr. Öğr. Üyesi Ekrem Yaşar**

Kursun geldiği yer:

| Oturum | Sistem | Ölçek |
|---|---|---|
| 2 | Suda 1 lizozim | ~5 nm |
| 3 | Membranda 1 GPCR (all-atom) | ~10 nm |
| 4 | Membranda 1 GPCR (Martini) | ~12 nm |
| **6** | **Kalabalık sistem** | **40–100 nm** |

Kursun adı *"büyük ve kalabalık hücresel sistemler"* — işte o kısım burası.


---
## Neden kalabalık?

Simülasyonlarımızın çoğu proteini **seyreltik çözeltide** inceler. Hücre öyle değil:

- Sitoplazmada protein derişimi **~300 g/L** — hacmin %20–30'u makromolekül
- Membranda proteinler birbirine değecek yoğunlukta
- Kalabalık **difüzyonu yavaşlatır**, **bağlanma dengelerini kaydırır**

**Sorun:** 100 proteini elle, çakışmadan, doğru yönelimle yerleştirmek imkânsız.
**Cevap:** [`bentopy`](https://github.com/marrink-lab/bentopy)

| Aşama | Komut | Ne yapar |
|---|---|---|
| 1 | `bentopy pack` | yapıları çakışmadan kutuya yerleştirir |
| 2 | `bentopy render` | plandan gerçek koordinatları üretir |
| 3 | `bentopy solvate` | boşluğu su ve iyonla doldurur |


---
## 1 · Kurulum


In [ ]:
%%capture
!pip install -q bentopy
!apt-get -qq update && apt-get -qq install -y gromacs


In [ ]:
!bentopy --help 2>&1 | head -20


---
## 2 · Girdi: Oturum 4'ün kaba-taneli AT2R'si

Oturum 4'te ürettiğimiz `at2r_cg.pdb`'yi kullanacağız.
Elinizde yoksa aşağıdaki hücre kurs deposundan hazır olanı indirir.


In [ ]:
import os
if not os.path.exists('at2r_cg.pdb'):
    !git clone -q https://github.com/eygpcr/biyofizik2026-martini.git repo
    !cp repo/04_martini_input/cikti/at2r_cg.pdb . 2>/dev/null || echo 'Hazir dosya bulunamadi'
!ls -la at2r_cg.pdb 2>/dev/null || print('Oturum 4 notebookundan at2r_cg.pdb dosyasini yukleyin')


In [ ]:
# GRO formatina cevir (bentopy .gro bekler)
!gmx editconf -f at2r_cg.pdb -o at2r_cg.gro -d 0.5 2>&1 | tail -5


---
## 3 · Yerleşim tanımı (`yerlesim.json`)

bentopy'ye ne istediğimizi bir JSON dosyasıyla anlatıyoruz:
kutu ne kadar büyük, hangi yapıdan kaç tane, ne kadar sıkı yerleşsin.

> 💡 `number` değerini artırmak kalabalığı artırır — ama paketleme süresi de uzar.


In [ ]:
import json

yerlesim = {
    'space': {
        'size': [40, 40, 20],        # kutu boyutu (nm)
        'resolution': 0.5,           # paketleme izgara cozunurlugu (nm)
        'compartments': [
            {'id': 'kutu', 'shape': 'cuboid'}
        ],
    },
    'output': {'title': 'Kalabalik AT2R membrani', 'topol_includes': []},
    'segments': [
        {
            'name': 'AT2R',
            'number': 25,            # kac kopya
            'path': 'at2r_cg.gro',
            'compartments': ['kutu'],
        }
    ],
}

json.dump(yerlesim, open('yerlesim.json','w'), indent=2)
print(open('yerlesim.json').read())


---
## 4 · `bentopy pack` — yerleşim planını üret

Bu aşamada henüz koordinat üretilmiyor; sadece *neyin nereye* konacağı hesaplanıyor.


In [ ]:
!bentopy pack yerlesim.json -o plan.json 2>&1 | tail -20


---
## 5 · `bentopy render` — koordinatları üret


In [ ]:
!bentopy render plan.json -o kalabalik.gro -t kalabalik.top 2>&1 | tail -20

import os
if os.path.exists('kalabalik.gro'):
    n = int(open('kalabalik.gro').read().splitlines()[1])
    print(f'\nToplam parcacik: {n:,}')


---
## 6 · `bentopy solvate` — boşluğu doldur


In [ ]:
!bentopy solvate -f kalabalik.gro -o sistem_kalabalik.gro 2>&1 | tail -20


---
## 7 · Ne kurduk?

Ölçeklerin karşılaştırması:


In [ ]:
import os
def parcacik(p):
    try: return int(open(p).read().splitlines()[1])
    except Exception: return None

for ad, dosya in [('Oturum 4 (tek GPCR)', 'sistem.gro'),
                  ('Oturum 6 (kalabalik)', 'sistem_kalabalik.gro')]:
    n = parcacik(dosya) if os.path.exists(dosya) else None
    print(f'{ad:<26}: {n:,} parcacik' if n else f'{ad:<26}: (dosya yok)')


---
## 🆘 Takıldıysanız — sorun değil

`bentopy` kurulumu bazen sürprizli olabiliyor ve 40 dakikamız var.
Takılırsak zaman kaybetmeden geçiyoruz; elinizde kalanlar:

- ✅ Tüm komutlar ve JSON şablonu → [`06_bentopy/kodlar/`](https://github.com/eygpcr/biyofizik2026-martini/tree/main/06_bentopy/kodlar)
- ✅ Baştan sona uygulama videosu → [`VIDEO.md`](https://github.com/eygpcr/biyofizik2026-martini/blob/main/06_bentopy/VIDEO.md)
- ✅ Hazır çıktı dosyaları

---

## 📚 Buradan sonrası

- [bentopy resmî tutorial'ı](https://cgmartini.nl/docs/tutorials/Martini3/Bentopy/)
- [bentopy GitHub](https://github.com/marrink-lab/bentopy)
- [Protein kompleksleri — Martini](https://cgmartini.nl/docs/tutorials/Martini3/ProteinsIIb/) — oligomerizasyon analizi
- [TS2CG v2.0](https://github.com/weria-pezeshkian/TS2CG-v2.0/wiki/Tutorial) — vezikül, tübül, karmaşık şekiller

Ve tabii: [`ILERI_OKUMA.md`](https://github.com/eygpcr/biyofizik2026-martini/blob/main/ILERI_OKUMA.md)
